In [55]:
import asyncio
import logging
import re
from google.genai import types

import requests

from google.adk.agents import LlmAgent
from google.adk.models import LlmResponse
from google.adk.tools import google_search

# -------------
# -- Logging --

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
    force=True,
)
logger = logging.getLogger("weather_app")

# -----------------------------
# -- Weather retrieval (NWS) --

def get_nws_weather(latitude: float, longitude: float) -> dict:
    base_url = "https://api.weather.gov"
    headers = {"User-Agent": "MyWeatherApp (tmontgomery)", "Accept": "application/json"}

    try:
        points_url = f"{base_url}/points/{latitude},{longitude}"
        points_response = requests.get(points_url, headers=headers, timeout=10)
        points_response.raise_for_status()
        points_data = points_response.json()

        city = (
            points_data.get("properties", {})
            .get("relativeLocation", {})
            .get("properties", {})
            .get("city")
        )
        state = (
            points_data.get("properties", {})
            .get("relativeLocation", {})
            .get("properties", {})
            .get("state")
        )

        forecast_url = points_data.get("properties", {}).get("forecast")
        if not forecast_url:
            return {}

        forecast_response = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_response.raise_for_status()
        forecast_data = forecast_response.json()

        periods = forecast_data.get("properties", {}).get("periods", [])
        if not periods:
            return {}

        current_period = periods[0]
        temperature = f"{current_period.get('temperature')} {current_period.get('temperatureUnit')}"
        detailed_forecast = current_period.get("detailedForecast")

        return {
            "coordinates": {"Latitude": latitude, "Longitude": longitude},
            "city": city or "Unknown City",
            "state": state or "Unknown State",
            "temperature": temperature or "N/A",
            "detailed_forecast": detailed_forecast or "No detailed forecast available.",
        }

    except requests.exceptions.RequestException as e:
        logger.exception("Error fetching weather data: %s", e)
        return {}

def get_extended_weather_forecast(latitude: float, longitude: float) -> str:
    data = get_nws_weather(latitude, longitude)
    if not data:
        return "Unable to retrieve weather right now."

    return (
        f"Location: {data['city']}, {data['state']}\n"
        f"Coordinates: {data['coordinates']}\n"
        f"Temperature: {data['temperature']}\n"
        f"Forecast: {data['detailed_forecast']}"
    )

# -------------------------------------------
# -- Moderation + callbacks (WeatherAgent) --

_ALLOWED_CHARS = re.compile(r"^[0-9\s\-\+\,\.\(\):=a-zA-Z]*$")
_FLOAT = r"[-+]?\d+(?:\.\d+)?"
_LAT_LON_2FLOATS = re.compile(rf"({_FLOAT})\s*[, ]\s*({_FLOAT})")
_LAT_EQ = re.compile(rf"\blat(?:itude)?\s*[:=]\s*({_FLOAT})\b", re.IGNORECASE)
_LON_EQ = re.compile(rf"\b(lon|lng|longitude)\s*[:=]\s*({_FLOAT})\b", re.IGNORECASE)

def _extract_last_user_text(llm_request) -> Optional[str]:
    if not getattr(llm_request, "contents", None):
        return None
    last = llm_request.contents[-1]
    if getattr(last, "role", None) != "user":
        return None
    parts = getattr(last, "parts", None) or []
    if not parts or not getattr(parts[0], "text", None):
        return None
    return parts[0].text.strip()

def _parse_lat_lon(text: str) -> Tuple[float, float]:
    mlat = _LAT_EQ.search(text)
    mlon = _LON_EQ.search(text)
    if mlat and mlon:
        return float(mlat.group(1)), float(mlon.group(2))

    m = _LAT_LON_2FLOATS.search(text)
    if m:
        return float(m.group(1)), float(m.group(2))

    raise ValueError("Could not find latitude/longitude.")

def _is_in_us_approx(lat: float, lon: float) -> bool:
    in_conus = 24.396308 <= lat <= 49.384358 and -124.848974 <= lon <= -66.885444
    in_ak    = 51.2      <= lat <= 71.5      and -179.15     <= lon <= -129.99
    in_hi    = 18.9      <= lat <= 22.3      and -160.3      <= lon <= -154.8
    return in_conus or in_ak or in_hi

def _reject(message: str):
    return LlmResponse(content={"role": "model", "parts": [{"text": message}]})

def log_user_prompt(callback_context, llm_request):
    user_text = _extract_last_user_text(llm_request)
    if user_text:
        logger.info("[%s] USER » %s", callback_context.agent_name, user_text)
    return None

def log_model_response(callback_context, llm_response):
    content = getattr(llm_response, "content", None)
    parts = getattr(content, "parts", None) if content else None
    if parts and getattr(parts[0], "text", None):
        logger.info("[%s] MODEL » %s", callback_context.agent_name, parts[0].text.strip())
    return None

def moderate_user_prompt(callback_context, llm_request):
    user_text = _extract_last_user_text(llm_request)
    if not user_text:
        return None

    if len(user_text) > 200:
        return _reject("Provide coordinates like: 34.6272, -78.6107.")
    if not _ALLOWED_CHARS.fullmatch(user_text):
        return _reject("Only latitude/longitude input is allowed.")

    try:
        lat, lon = _parse_lat_lon(user_text)
    except Exception:
        return _reject("Unable to process coordinates. Use: 34.6272, -78.6107.")

    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        return _reject("Latitude/longitude out of range.")
    if not _is_in_us_approx(lat, lon):
        return _reject("Location must be within the US.")

    return None

def chained_before_callback(callback_context, llm_request):
    moderation_result = moderate_user_prompt(callback_context, llm_request)
    if moderation_result is not None:
        return moderation_result
    return log_user_prompt(callback_context, llm_request)

# -----------------
# -- Search tool --

#def _load_google_search_tool():
#  from google.adk.tools import google_search
#GoogleSearchTool = _load_google_search_tool()

def _load_google_search_tool():
    try:
        from google.adk.tools import google_search as Tool
        return Tool
    except Exception:
        from google.adk.tools import google_search_tool as Tool
        return Tool

GoogleSearchTool = _load_google_search_tool()



# ------------
# -- Agents --

WEATHER_AGENT_INSTRUCTIONS = (
    "You are a friendly weather agent. "
    "If the user provides US latitude/longitude, call the weather tool and answer concisely. "
    "If coordinates are missing, ask for them."
)

SEARCH_AGENT_INSTRUCTIONS = (
    "You are a search specialist. Use the Google Search tool to answer. "
    "Return a concise summary."
)

ROOT_AGENT_INSTRUCTIONS = (
    "You are the coordinator (root agent). "
    "Delegates to WeatherAgent for weather from US latitude/longitude. "
    "Delegates to SearchAgent for general web lookup. "
    "Delegates to exactly one sub-agent unless the user explicitly asks for both."
)

weather_agent = LlmAgent(
    name="WeatherAgent",
    model="gemini-2.0-flash",
    description="Gets NWS weather for US latitude/longitude.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_extended_weather_forecast],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

search_agent = LlmAgent(
    name="SearchAgent",
    model="gemini-2.0-flash",
    description="Finds answers using Google Search.",
    instruction=SEARCH_AGENT_INSTRUCTIONS,
    tools=[],
)

def _build_root_agent():
    return LlmAgent(
      name="RootAgent",
      model="gemini-2.0-flash",
      description="Delegates to weather or search specialists.",
      instruction=ROOT_AGENT_INSTRUCTIONS,
      sub_agents=[weather_agent, search_agent],
    )

root_agent = _build_root_agent()

# ------------------------
# -- Session and Runner --

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

APP_NAME = "David"
USER_ID = "user123"
SESSION_ID = "session456"

async def setup_session_and_runner():
  session_service = InMemorySessionService()
  session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
  runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)
  return session, runner


async def run_and_print_events(user_text: str, session_id: str = SESSION_ID):
    print("\n" + "=" * 90)
    print("USER:", user_text)
    print("=" * 90)
    session, runner = await setup_session_and_runner()
    event_stream = runner.run_async(user_id="user123", session_id=session_id)

async def call_agent_async(query):
    content = types.Content(role='user', parts=[types.Part(text=query)])
    session, runner = await setup_session_and_runner()
    events = runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content)

    async for event in event_stream:
        event_type = getattr(event, "type", event.__class__.__name__)
        agent_name = getattr(event, "agent_name", None)

        text = None
        content = getattr(event, "content", None)
        if isinstance(content, dict):
            parts = content.get("parts") or []
            if parts and isinstance(parts, list) and isinstance(parts[0], dict):
                text = parts[0].get("text")

        print(f"event type={event_type} agent={agent_name} text={text}")

async def demo():
    await run_and_print_events("34.6272, -78.6107")
    await run_and_print_events("Search: What is Google ADK multi-agent?")

await demo()



USER: 34.6272, -78.6107

USER: Search: What is Google ADK multi-agent?
